# Download and store STOOQ data

In [34]:
import warnings
warnings.filterwarnings('ignore')

In [35]:

import requests
from io import BytesIO
from zipfile import ZipFile, BadZipFile

import numpy as np
import pandas as pd
import pandas_datareader.data as web
from sklearn.datasets import fetch_openml
import os
import glob
import h5py as h5
path='/Users/Massimiliano'
os.chdir(path)

pd.set_option('display.expand_frame_repr', False)

In [36]:
DATA_STORE = 'assets1.h5'

In [37]:
stooq_path = '/Users/Massimiliano/'


In [38]:
metadata_dict = {
    ('us', 'nasdaq etfs'): 69,
    ('us', 'nasdaq stocks'): 27,
    ('us', 'nyse etfs'): 70,
    ('us', 'nyse stocks'): 28,
    ('us', 'nysemkt stocks'): 26
}

In [43]:
import glob
import os
import pandas as pd

dfs = {}

for (market, asset_class), code in metadata_dict.items():
    path = f'/Users/Massimiliano/stooq/us/{code}'
    files = glob.glob(f'{path}/**/*.txt', recursive=True)
    dfs[code] = pd.concat([pd.read_csv(f, engine='python', header=0)
                        .astype(str)
                        .apply(lambda x: x.str.strip())
                        .dropna()
                        .assign(asset_class=asset_class)
                        .reset_index(drop=True)
                        for f in files if os.stat(f).st_size != 0],
                        ignore_index=True)
    dfs[code] = dfs[code].drop_duplicates(subset='<TICKER>')
    print(f'{asset_class} ({code}): {dfs[code].shape[0]:,.0f} rows')
    output_path = "/Users/Massimiliano/"
    output_file = f'{asset_class}.csv'
    output_full_path = os.path.join(output_path, output_file)
    dfs[code]['<TICKER>'].to_csv(output_full_path, index=False)

nasdaq etfs (69): 366 rows
nasdaq stocks (27): 4,745 rows
nyse etfs (70): 2,163 rows
nyse stocks (28): 3,738 rows
nysemkt stocks (26): 359 rows


In [15]:
# Load CSV files into a dictionary of DataFrames
dfs = {}
for asset_class in ['nasdaq etfs','nasdaq stocks', 'nyse etfs', 'nyse stocks', 'nysemkt stocks']:
    filename = f'/Users/Massimiliano/{asset_class}.csv'
    dfs[asset_class] = pd.read_csv(filename)

# Extract tickers and prices from the dfs dictionary
tickers = []
prices = []
for asset_class, df in dfs.items():
    print(asset_class, df.columns)
    tickers.extend(df['<TICKER>'].tolist())
    prices.append(df.set_index(['<TICKER>', '<DATE>'])['<CLOSE>'].unstack('<TICKER>'))


# Combine prices into a single DataFrame
prices = pd.concat(prices, axis=1)

# Drop duplicates from prices DataFrame
prices = prices.loc[:, ~prices.columns.duplicated()]
prices = prices.loc[~prices.index.duplicated(keep='last')]

# Save tickers and prices to HDF5 file
with h5.File('/Users/Massimiliano/assets1.h5', 'w') as f:
    f.create_dataset('tickers', data=np.array(tickers, dtype='S'))
    f.create_dataset('prices', data=prices.values, compression='gzip')
    f.create_dataset('columns', data=prices.columns.values, compression='gzip')
    f.create_dataset('index', data=prices.index.values, compression='gzip')

nasdaq etfs Index(['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>',
       '<CLOSE>', '<VOL>', '<OPENINT>', 'asset_class'],
      dtype='object')
nasdaq stocks Index(['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>',
       '<CLOSE>', '<VOL>', '<OPENINT>', 'asset_class'],
      dtype='object')
nyse etfs Index(['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>',
       '<CLOSE>', '<VOL>', '<OPENINT>', 'asset_class'],
      dtype='object')
nyse stocks Index(['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>',
       '<CLOSE>', '<VOL>', '<OPENINT>', 'asset_class'],
      dtype='object')
nysemkt stocks Index(['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>',
       '<CLOSE>', '<VOL>', '<OPENINT>', 'asset_class'],
      dtype='object')


### Store price data in HDF5 format

In [30]:

def printGroup(group):
    for i in list(group. keys()):
        try:
            if list(group[i].keys()):
                print(f"{group.name}/{i}/")
                printGroup(group[i])
        except:
            print(group[i].name, group[i].dtype, group[i].shape)

print("\n\nData File Structure")
f = h5.File('/Users/Massimiliano/assets1.h5', 'r')
printGroup(f)



Data File Structure
/columns object (11371,)
/index int64 (15547,)
/prices float64 (15547, 11371)
/tickers |S12 (24456286,)


To speed up loading, we store the price data in HDF format. The function `get_stooq_prices_and_symbols` loads data assuming the directory structure described above and takes the following arguments:
- frequency (see Stooq website for options as these may change; default is `daily`
- market (default: `us`), and 
- asset class (default: `nasdaq etfs`.

It removes files that do not have data or do not appear in the corresponding list of symbols.

In [18]:
def get_stooq_prices_and_tickers(frequency='daily',
                                 market='us',
                                 asset_class='nasdaq etfs'):
    prices = []
    
    tickers = (pd.read_csv('/Users/Massimiliano/' f'{asset_class}.csv'))

    if frequency in ['5 min', 'hourly']:
        parse_dates = [['date', 'time']]
        date_label = 'date_time'
    else:
        parse_dates = ['date']
        date_label = 'date'
    names = ['ticker', 'freq', 'date', 'time', 
             'open', 'high', 'low', 'close','volume', 'openint']
    
    usecols = ['ticker', 'open', 'high', 'low', 'close', 'volume'] + parse_dates
    path = stooq_path / 'data' / frequency / market / asset_class
    print(path.as_posix())
    files = path.glob('**/*.txt')
    for i, file in enumerate(files, 1):
        if i % 500 == 0:
            print(i)
        if file.stem not in set(tickers.ticker.str.lower()):
            print(file.stem, 'not available')
            file.unlink()
        else:
            try:
                df = (pd.read_csv(
                    file,
                    names=names,
                    usecols=usecols,
                    header=0,
                    parse_dates=parse_dates))
                prices.append(df)
            except pd.errors.EmptyDataError:
                print('\tdata missing', file.stem)
                file.unlink()

    prices = (pd.concat(prices, ignore_index=True)
              .rename(columns=str.lower)
              .set_index(['ticker', date_label])
              .apply(lambda x: pd.to_numeric(x, errors='coerce')))
    return prices, tickers

We'll be using US equities and ETFs in [Chapter 9](../09_time_series_models) and and Japanese equities in [Chapter 11](../11_decision_trees_random_forests). The following code collects the price data for the period 2000-2019 and stores it with the corresponding symbols in the global `assets.h5` store:

In [19]:
# load some Japanese and all US assets for 2000-2019
markets = {'us': ['nasdaq etfs', 'nasdaq stocks', 'nyse etfs', 'nyse stocks', 'nysemkt stocks']
        }
frequency = 'daily'

idx = pd.IndexSlice
for market, asset_classes in markets.items():
    for asset_class in asset_classes:
        print(f'\n{asset_class}')
        prices, tickers = get_stooq_prices_and_tickers(frequency=frequency, 
                                                       market=market, 
                                                       asset_class=asset_class)
        
        prices = prices.sort_index().loc[idx[:, '2000': '2019'], :]
        names = prices.index.names
        prices = (prices
                  .reset_index()
                  .drop_duplicates()
                  .set_index(names)
                  .sort_index())
        
        print('\nNo. of observations per asset')
        print(prices.groupby('ticker').size().describe())
        key = f'stooq/{market}/{asset_class.replace(" ", "/")}/'
        
        print(prices.info(null_counts=True))
        
        prices.to_hdf(DATA_STORE, key + 'prices', format='t')
        
        print(tickers.info())
        tickers.to_hdf(DATA_STORE, key + 'tickers', format='t')


nasdaq etfs


TypeError: unsupported operand type(s) for /: 'str' and 'str'